# ISL Static Sign Recognition (MediaPipe) - compact version
Fixes: mirror/handedness mismatch (main cause of C -> O), extra finger-distance features, mirror augmentation, and smoothed live predictions.

In [1]:
%pip install -q mediapipe opencv-python pandas numpy scikit-learn joblib tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import zipfile, joblib, cv2, numpy as np, pandas as pd, mediapipe as mp
from collections import deque, Counter
from tqdm.auto import tqdm

PROJECT_DIR = Path(r"C:\AIML 5 D\Study material\Introduction to Deep Learning\mini\mini project")
ZIP_PATH, EXTRACT_DIR = PROJECT_DIR / "ISL dataset.zip", PROJECT_DIR / "isl dataset"
OUT_DIR = PROJECT_DIR / "isl_mediapipe_output"; OUT_DIR.mkdir(exist_ok=True)
CSV_PATH, MODEL_PATH = OUT_DIR / "isl_landmarks.csv", OUT_DIR / "isl_model.joblib"

mp_hands = mp.solutions.hands
TIPS, IU = [4, 8, 12, 16, 20], np.triu_indices(5, 1)

def hand_to_features(hand):
    """21 landmarks -> 63 values, wrist-centred and scale-normalised."""
    pts = np.array([[l.x, l.y, l.z] for l in hand.landmark], np.float32)
    pts -= pts[0]
    return (pts / max(np.linalg.norm(pts[:, :2], axis=1).max(), 1e-6)).ravel()

def extract(image_bgr, hands_model):
    """Returns (126-vector [Left, Right] or None, mediapipe result)."""
    res = hands_model.process(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    if not res.multi_hand_landmarks:
        return None, res
    slots = [np.zeros(63, np.float32), np.zeros(63, np.float32)]
    for lm, hd in zip(res.multi_hand_landmarks, res.multi_handedness):
        i = 0 if hd.classification[0].label == "Left" else 1
        if slots[i].any():
            i = 1 - i                      # both labelled the same -> use the free slot
        slots[i] = hand_to_features(lm)
    return np.concatenate(slots), res

def mirror(X):
    """Feature-space horizontal flip: negate x and swap Left/Right slots."""
    P = X.reshape(-1, 2, 21, 3).copy()
    P[..., 0] *= -1
    return P[:, ::-1].reshape(len(X), -1)

def add_features(X):
    """Append fingertip-to-fingertip and fingertip-to-wrist distances (key for C vs O)."""
    T = X.reshape(-1, 2, 21, 3)[:, :, TIPS]
    d = np.linalg.norm(T[:, :, :, None] - T[:, :, None], axis=-1)[:, :, IU[0], IU[1]]
    w = np.linalg.norm(T, axis=-1)
    return np.hstack([X, d.reshape(len(X), -1), w.reshape(len(X), -1)])

# ---- Build (or reload) the landmark table ----
if CSV_PATH.exists():
    df = pd.read_csv(CSV_PATH)
else:
    if not EXTRACT_DIR.exists():
        zipfile.ZipFile(ZIP_PATH).extractall(EXTRACT_DIR)
    data_dir = next(p for p in EXTRACT_DIR.rglob("Indian") if p.is_dir())
    paths = [f for f in data_dir.rglob("*") if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}]
    rows = []
    with mp_hands.Hands(static_image_mode=True, max_num_hands=2, model_complexity=1,
                        min_detection_confidence=0.5) as hands:
        for p in tqdm(paths):
            img = cv2.imread(str(p))
            if img is None:
                continue
            img = cv2.resize(img, (256, 256), interpolation=cv2.INTER_CUBIC)
            feats, _ = extract(img, hands)
            if feats is not None:
                rows.append([p.parent.name, *feats])
    df = pd.DataFrame(rows, columns=["label"] + [f"f{i}" for i in range(126)])
    df.to_csv(CSV_PATH, index=False)
print(df.shape); print(df.label.value_counts().sort_index().to_dict())

AttributeError: module 'mediapipe' has no attribute 'solutions'

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report

X = df.drop(columns="label").to_numpy(np.float32)
le = LabelEncoder(); y = le.fit_transform(df.label.astype(str))
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Mirror augmentation: model learns signs from both camera orientations.
Xtr, ytr = np.vstack([Xtr, mirror(Xtr)]), np.concatenate([ytr, ytr])

model = make_pipeline(StandardScaler(),
                      MLPClassifier((256, 128), early_stopping=True, max_iter=300, random_state=42))
model.fit(add_features(Xtr), ytr)

pred = model.predict(add_features(Xte))
print(f"Test accuracy: {accuracy_score(yte, pred):.4f}")
print(f"Mirrored-test accuracy: {accuracy_score(yte, model.predict(add_features(mirror(Xte)))):.4f}")
print(classification_report(yte, pred, target_names=le.classes_, zero_division=0))
if {"C", "O"} <= set(le.classes_):
    c, o = le.transform(["C", "O"])
    print(f"C predicted as O: {((yte == c) & (pred == o)).sum()} of {(yte == c).sum()}")

joblib.dump({"model": model, "classes": le.classes_}, MODEL_PATH)

In [ ]:
bundle = joblib.load(MODEL_PATH)
model, classes = bundle["model"], bundle["classes"]

CAMERA = 0   # or e.g. "http://192.168.29.117:4747/video"
cap = cv2.VideoCapture(CAMERA, cv2.CAP_DSHOW) if isinstance(CAMERA, int) else cv2.VideoCapture(CAMERA)
history, draw = deque(maxlen=10), mp.solutions.drawing_utils

try:
    with mp_hands.Hands(static_image_mode=False, max_num_hands=2, model_complexity=1,
                        min_detection_confidence=0.6, min_tracking_confidence=0.6) as hands:
        while cap.isOpened():
            ok, frame = cap.read()
            if not ok:
                break
            feats, res = extract(frame, hands)          # raw frame; mirror augmentation covers flips
            text = "No hand detected"
            if feats is not None:
                for h in res.multi_hand_landmarks:
                    draw.draw_landmarks(frame, h, mp_hands.HAND_CONNECTIONS)
                probs = model.predict_proba(add_features(feats[None]))[0]
                best = int(probs.argmax())
                history.append(classes[best] if probs[best] > 0.5 else "?")
                text = f"{Counter(history).most_common(1)[0][0]} ({probs[best]:.0%})"
            else:
                history.clear()
            frame = cv2.flip(frame, 1)                  # mirror view for display only
            cv2.putText(frame, text, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)
            cv2.imshow("ISL Live - press Q to quit", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
finally:
    cap.release(); cv2.destroyAllWindows()